# 習題：數學物件、Python 型別與 AI 中的數值表示

在數學中，我們經常接觸下列概念：

- 自然數（Natural Numbers）
- 整數（Integers）
- 有理數（Rational Numbers）
- 實數（Real Numbers）
- 複數（Complex Numbers）
- 向量（Vectors）
- 矩陣（Matrices）
- 集合（Sets）
- 命題的真假（Truth Values）

本習題將探討：

```text
數學物件
    ↓
Python 型別
    ↓
CPython 實作
    ↓
AI 中的數值表示
```

---

## 第一部分：Python 如何表示數學物件？

請查閱 Python 文件或自行測試，完成下表。

| 數學概念 | Python 型別 | 範例 |
|----------|------------|------|
| 自然數 | int | 5 |
| 整數 | int | -5 |
| 有理數 | fractions.Fraction | Fraction(1, 3) |
| 實數 | float | 3.14159 |
| 複數 | complex | 3 + 4j |
| 向量 | list | [1, 2, 3] |
| 矩陣 | list | [[1, 2], [3, 4]] |
| 集合 | set | {1, 2, 3} |
| 命題真假 | bool | True |

並回答下列問題：

1. 哪些是 Python 的內建型別（Built-in Types）？
int、float、complex、list、set、bool。
2. 哪些需要額外模組才能較完整地表示？
有理數：需要標準函式庫 fractions 模組中的 Fraction 型別。

向量與矩陣：雖然可以使用內建的 list 表示，但若需要完整且高效的線性代數運算，需要使用第三方庫 numpy 的 ndarray。
3. Python 的型別分類與數學中的分類是否完全一致？

不完全一致。例如：數學上的自然數 $\mathbb{N}$ 與整數 $\mathbb{Z}$ 在 Python 中都被併入 int 型別。數學上的實數 $\mathbb{R}$ 包含無理數且具連續性，但 Python 的 float 是基於 IEEE 754 標準的雙精度浮點數，無法精確表示所有實數（如 $\pi$ 或不循環小數）。Python 的 bool 是 int 的子類別（True 的值為 1，False 的值為 0）。

---

## 第二部分：數學概念與 Python 型別

數學上有下列包含關係：

$$
\mathbb N
\subseteq
\mathbb Z
\subseteq
\mathbb Q
\subseteq
\mathbb R
\subseteq
\mathbb C
$$

請回答：

1. Python 是否為上述每一個數系都提供獨立型別？
否。Python 沒有獨立的自然數型別，且有理數需要透過 fractions 模組，而非語言本身的內建獨立關鍵字型別。
2. 為什麼自然數與整數都使用 `int` 表示？
自然數是整數的子集（非負整數）。在程式設計中，為自然數獨立設計一個型別會增加型別檢查與轉換的複雜度（例如兩自然數相減可能得到負整數）。統一使用大範圍的任意精度整數 int 能簡化語言設計與使用體驗。
3. Python 為什麼提供 `complex` 型別？
複數在科學計算、訊號處理、工程學與物理學中廣泛應用。Python 初衷是作為易用的科學計算與通用語言，內建 complex 支援能讓使用者直接進行複數算術，毋需額外定義類別。
4. Python 為什麼沒有內建的有理數型別？
浮點數（float）能滿足絕大多數日常計算需求且具備硬體加速效能。有理數（分數）運算需維護分子與分母，涉及頻繁的最大公因數（GCD）約分，計算開銷較大。因此 Python 將其放在標準模組 fractions 中，以供有嚴格精確度需求時按需調用。
5. 下列三個物件的型別有何不同？
1：型別為 int，表示整數。

1.0：型別為 float，表示雙精度浮點數。

1+0j：型別為 complex，表示複數（虛部為 0）。

```python
1
1.0
1+0j
```

---

## 第三部分：CPython 如何實作這些型別？

除了知道 Python 使用哪些型別表示數學物件之外，

我們也希望了解：

> Python 物件在 CPython 中是如何實作的？

請查閱資料，完成下表。

| 數學概念 | Python 型別 | CPython 實作 | 主要特點 |
|----------|------------|-------------|----------|
| 自然數 | `int` | PyLongObject | 動態陣列儲存數位（digit），實現任意精度 |
| 整數 | `int` | PyLongObject | 動態陣列儲存數位，長度欄位同時編碼數值的正負號 |
| 實數 | `float` | PyFloatObject | 包裝 C 語言的 double（IEEE 754 64位元浮點數） |
| 複數 | `complex` | PyComplexObject | 包含兩個 double 欄位（real 與 imag） |
| 集合 | `set` | PySetObject | 基於哈希表（Hash Table）實作，去重且無序 |
| 命題真假 | `bool` | PyLongObject | int 的子類別，內部全域單例 Py_True 與 Py_False |
| 串列 | `list` | PyListObject | 動態陣列（Array of pointers），元素儲存物件指標 |

---

## 第四部分：深入探討各種型別

請針對第三部分中的每一種型別回答：

1. 它是否為 Python 的內建型別？
表格中的所有型別（int、float、complex、set、bool、list）均為 Python 內建型別。
2. 對應的 CPython 物件結構名稱為何？
3. 它如何儲存資料？
4. 是否有固定大小限制？
5. 若資料規模持續增加，如何擴充儲存空間？

int (PyLongObject)：
儲存方式：變長物件（VarObject）。內部維護一個變長陣列（ob_digit），以 30 位元（或 15 位元）為一個 digit 單位進行分段儲存。
大小限制與擴充：無硬體位元數限制，僅受限於記憶體大小；資料規模增加時動態重分配陣列空間。

float (PyFloatObject)：
儲存方式：定長物件。直接包裹 C 語言的 double 型別（64 位元 IEEE 754 標準，包含 1 位元正負號、11 位元指數、52 位元小數部分）。
大小限制與擴充：有固定大小限制（8 位元組），無法動態擴充。超過範圍會引發 OverflowError。

complex (PyComplexObject)：
儲存方式：定長物件。包含兩個 double 變數 (Py_complex 結構，包含 real 和 imag)。
大小限制與擴充：有固定大小限制（16 位元組），無法動態擴充。

set (PySetObject)：
儲存方式：變長物件。基於哈希表實作，內部維護 key-hash 陣列。
大小限制與擴充：無固定大小限制；當負載因子過高時，會動態重新擴容（resize）並重哈希。

bool (PyLongObject 子類) ：
儲存方式：本質上是 int，CPython 中僅預先創建兩個全域實例 Py_True 與 Py_False。
大小限制與擴充：固定為單例物件，不可擴充。

list (PyListObject)：
儲存方式：變長物件。維護一個連續的 C 指標陣列（PyObject** ob_item）。
大小限制與擴充：無固定限制；當元素增加超過容量時，會採取過度分配策略（Over-allocation）重新分配更大的記憶體塊。
---

## 第五部分：Python 整數與浮點數

執行下列程式：

```python
print(10**100)
print(10**1000)
```

回答：

1. 為什麼 Python 能表示如此大的整數？
因為 Python 的整數採用大數運算（Arbitrary Precision Arithmetic）。它將大整數拆分成多個小區塊儲存在記憶體陣列中，透過軟體層面實現多位數的加減乘除。
2. Python 的 `int` 是否具有固定的位元數限制？
沒有固定的位元數限制（例如 C 語言中的 32 位或 64 位）。
3. 什麼是 Arbitrary Precision（任意精度）？
指數值的大小與精度只受到電腦可用記憶體（RAM）的限制，而不會因為固定位元數上限而產生溢位（Overflow）。
4. CPython 使用什麼物件來實作 `int`？
使用 PyLongObject 結構。
5. Python 整數大小真正的限制是什麼？
系統的可利用實體與虛擬記憶體大小，以及 CPython 內部對陣列長度（ob_size）的上限限制（通常為 $2^{63}-1$ 個 digit，實質上等同於記憶體極限）。

---

再執行：

```python
0.1 + 0.2
```

```python
0.1 + 0.2 == 0.3
```

```python
format(0.1, ".20f")
```

回答：

1. 為什麼結果與數學上的答案不同？
因為電腦內部以二進位表示浮點數。十進位小數 $0.1$ 和 $0.2$ 轉換成二進位時是無限循環小數，以有限位元（64 位元）儲存時必須截斷，導致微小的截斷誤差，相加結果為 0.30000000000000004。
2. 浮點數是否能精確表示所有小數？
不能。只有可以寫成二進位有限分數（即分母為 $2^n$ 的組合，如 $0.5, 0.25$）的小數才能被二進位浮點數精確表示。
3. 什麼是浮點數的精度（precision）？
指數值能被精確紀錄的有效數字位數。IEEE 754 雙精度浮點數具備 53 位元的尾數（Significand），相當於十進位約 15 至 17 位的有效數字精度。
4. 什麼是浮點數的範圍（range）？
指數值能表示的最大值與最小值區間。IEEE 754 雙精度浮點數由 11 位元指數決定，範圍約為 $10^{-308}$ 至 $10^{308}$。
5. 精度與範圍有何不同？
範圍決定了數值可以多大或多小（由指數位元決定）；精度決定了數值在該範圍內能有多精細、有多精確（由尾數位元決定）。

---

## 第六部分：AI 模型中的數值表示

人工智慧模型需要儲存大量參數（Parameters）與權重（Weights）。

請查閱資料並回答：

1. AI 模型中的權重通常使用什麼型別儲存？
通常使用浮點數型別，如 FP32、FP16、BF16，以及量化後的整數型別如 INT8、INT4。
2. 什麼是 FP64？
雙精度浮點數（Double Precision, 64-bit），包含 1 位符號、11 位指數、52 位尾數
3. 什麼是 FP32？
單精度浮點數（Single Precision, 32-bit），包含 1 位符號、8 位指數、23 位尾數。
4. 什麼是 FP16？
半精度浮點數（Half Precision, 16-bit），包含 1 位符號、5 位指數、10 位尾數。
5. 什麼是 BF16？
Bfloat16（Brain Floating Point, 16-bit），包含 1 位符號、8 位指數、7 位尾數。與 FP32 擁有相同動態範圍，但尾數精度較低。
6. 為什麼大型 AI 模型不直接使用 Python 的 `int` 作為權重？
不支援連續微積分：AI 訓練需要透過梯度下降（Gradient Descent）更新權重，梯度通常是非常小的連續小數。

效能極低：Python 的 int 是軟體層面實作的物件，不支援 GPU 硬體原生的并行矩陣乘法加速。

---

## 第七部分：比較不同數值表示方式

完成下表。

| 表示方式 | 位元數 | 是否有誤差 | 主要用途 |
|----------|--------|------------|----------|
| int | 動態（32-bit+） | 無誤差（精確整數） | 通用邏輯控制、計數器、索引 |
| FP64 | 64 | 有 | 高精度科學計算、物理模擬、金融計算 |
| FP32 | 32 | 有 | 傳統 AI 模型訓練、圖形渲染、通用計算 |
| FP16 | 16 | 有 | AI 訓練與推論（需注意溢位問題） |
| BF16 | 16 | 有 | 現代深度學習/大語言模型訓練與推論 |

並回答：

1. 哪種表示方式最精確？
在上述浮點數中為 FP64；若僅限整數運算則 Python 的 int 具備完全精確性。
2. 哪種表示方式最節省記憶體？
FP16 與 BF16（皆僅佔用 16 位元/2 位元組）。
3. 哪種表示方式最常見於 AI 訓練？
BF16（現代主流）與 FP32 / 混合精度（FP16/FP32）。
4. 哪種表示方式最常見於 AI 推論（Inference）？
FP16、BF16 以及量化型別（如 INT8/INT4）。

---

## 第八部分：綜合思考

### 思考題一

Python 官方文件指出：

```python
int
```

具有 Unlimited Precision（無限制精度）。

請回答：

1. Unlimited Precision 是什麼意思？
指型別沒有固定位元寬度（如 32-bit 或 64-bit）限制，數值位數可根據需求動態增長，維持完全精確。
2. 它是否代表可以儲存無限大的整數？
否。理論上沒有邏輯上限，但實際上受限於實體電腦的記憶體容量與 CPU 處理時間。
3. CPython 如何實現這項特性？
透過 PyLongObject 結構體，將整數轉換成多個 digit 陣列，並透過 C 語言實現大數加減乘除算法進行計算。

---

### 思考題二

比較：

```python
int
```

與

```python
float
```

回答：

1. 為什麼 `int` 幾乎沒有固定上限？
因為採用了動態記憶體配置，當數值增大時，軟體會自動申請額外的記憶體空間來擴展位數。
2. 為什麼 `float` 卻存在範圍限制？
因為 float 的設計初衷是為了硬體加速，直接採用 IEEE 754 標準固定劃分 64 位元（8 位元組）記憶體空間，其指數位元（11 位元）決定了最高與最低範圍。
3. 為什麼 `float` 可能產生誤差？
因為十進位小數在轉換為有限位元（52 位元尾數）的二進位浮點數時，無法精確表示無窮循環二進位小數，必須進行捨入與截斷。

---

### 思考題三

比較下列兩種設計理念：

#### Python 的整數

```text
Arbitrary Precision
正確性優先
```

#### AI 模型常用的 FP16、BF16

```text
Limited Precision
效率優先
```

請說明：

1. 兩種設計理念的差異。
int 代表符號邏輯與精確性優先：不允許任何數值失真，寧可犧牲時間與空間效能。

FP16/BF16 代表效能與吞吐量優先：允許微小誤差，換取極致的計算速度與空間壓縮率。
2. 各自的優點與缺點。
int (任意精度)：
優點：完全精確，無 Overflow 風險。
缺點：記憶體開銷大，無法使用 GPU 硬體張量核心加速，不適用於小數運算。

FP16/BF16：
優點：佔用空間極小，硬體運算速度極快（Tensor Cores 加速），符合平行計算需求。
缺點：存在捨入誤差，FP16 容易發生數值下溢/溢位（Underflow/Overflow）。
3. 為什麼大型語言模型通常選擇較低精度的表示方式？
深度學習模型對微小數值噪聲具備高度容錯性（Robustness）。使用低精度（如 BF16）可將記憶體頻寬與顯存需求減半，大幅提升硬體矩陣運算吞吐量，從而使訓練與推論百億/千億參數模型成為可能。
4. 如果所有 AI 模型都改用 FP64，可能會產生哪些問題？
顯存崩潰：顯存需求提升 4 倍（相較於 FP16/BF16），極易導致 OOM（Out of Memory）。

算力浪費與頻寬瓶頸：GPU 計算單元（如 Tensor Cores）對 FP64 的吞吐量遠低於 FP16/BF16，且傳輸數據的頻寬開銷翻倍，導致訓練時間延長數倍且能源消耗巨大。

---

### 挑戰題

請用自己的話解釋：

```python
x = 10**1000
```

為什麼 Python 可以正常運作；

而大型 AI 模型卻通常選擇使用：

```text
FP32
FP16
BF16
```

而不是任意精度整數或高精度浮點數？

請從：

- 記憶體使用量
- 計算速度
- 數值精度
- GPU 運算效率

等角度進行討論。

Python 的整數是基於 CPython 的 PyLongObject 實現。當執行 10**1000 時，Python 會在記憶體中動態配置一塊空間，將這個 1001 位的超大整數拆分成多個 30-bit 的小單位（digits）陣列來儲存，並調用高精度算術演算法處理。這類計算屬於單核 CPU 的邏輯與精確運算，只要記憶體足夠，Python 就能正常運行。